### Importing Libraries 

In [13]:
import websocket 
import json 
import pandas as pd
import time
import numpy as np
import torch 
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
import torch.optim as optim

In [5]:
SYMBOL="btcusdt"
STREAM_URL=f"wss://stream.binance.com:9443/ws/{SYMBOL}@bookTicker"
MAX_TICKS=5000 # Number of rows to collect 

# Global list to store our processed data
tick_data=[]
start_time=time.time()


In [6]:
def on_message(ws,message):
    """Triggered every time Binance sends a new tick"""
    data= json.loads(message)

    # Extract raw level 2 Order Book Data 
    bid_price= float(data['b'])
    bid_qty=float(data['B'])
    ask_price= float(data['a'])
    ask_qty=float(data["A"])


    # --- Feature Engineering (On the fly) ---
    # 1.Mid-Price: The exact Center of the spread 
    mid_price= (bid_price + ask_price )/2.0

    # Order Book Imbalance (OBI): Pressure between buyers and sellers
    # Range is [-1 to 1]. Positive means more buying pressure, negative means selling pressure.
     
    obi = (bid_qty - ask_qty) / (bid_qty + ask_qty)

    tick_data.append({
        "timestamp":time.time(),
        "mid_price": mid_price,
        "obi": obi,
        "bid_qty": bid_qty,
        "ask_qty": ask_qty
        })
    
    # progress tracker
    if len(tick_data) % 500==0:
        print(f"collect {len(tick_data)}/{MAX_TICKS} ticks...")
    # stop condition
    if len(tick_data) >= MAX_TICKS:
        print(f"\n Target reached in {round(time.time() - start_time, 2 )} seconds. Closing connection...")

        ws.close()
    
def on_error(ws,error):
    print(f"Error: {error}")

def on_close(ws,close_status_code, close_msg):
    print("Connection closed.")

    df= pd.DataFrame(tick_data)
    df.to_csv("btc_hft_ticks.csv", index=False)
    print(f"Successfully saved {len(df)} rows to csv")
    print(df.head())

def on_open(ws):
    print(f"Connected to Binance {SYMBOL.upper()} stream. Waiting for ticks...")

if __name__ == "__main__":
    ws= websocket.WebSocketApp(
        STREAM_URL,
        on_open=on_open,
        on_message= on_message,
        on_error=on_error,
        on_close=on_close
    )
    ws.run_forever()

Connected to Binance BTCUSDT stream. Waiting for ticks...
collect 500/5000 ticks...
collect 1000/5000 ticks...
collect 1500/5000 ticks...
collect 2000/5000 ticks...
collect 2500/5000 ticks...
collect 3000/5000 ticks...
collect 3500/5000 ticks...
collect 4000/5000 ticks...
collect 4500/5000 ticks...
collect 5000/5000 ticks...

 Target reached in 71.27 seconds. Closing connection...
Connection closed.
Successfully saved 5000 rows to csv
      timestamp  mid_price       obi  bid_qty  ask_qty
0  1.782384e+09  61543.525 -0.846425  0.33913  4.07735
1  1.782384e+09  61543.525 -0.846415  0.33913  4.07707
2  1.782384e+09  61543.525 -0.846127  0.33982  4.07707
3  1.782384e+09  61543.525 -0.846089  0.33991  4.07707
4  1.782384e+09  61543.525 -0.846052  0.34000  4.07707


### Data Preprocessing and Target Labelling 

In [9]:
df=pd.read_csv("btc_hft_ticks.csv")

# Quant labeling with assistance of financial books to understand
LOOKAHEAD= 10

df["future_mid"]=df["mid_price"].shift(-LOOKAHEAD)
df["price_change"]= (df["future_mid"]-df["mid_price"])/ df["mid_price"]

# defining the volatility Threshold
# for ultra-fast ticks we are going to use the SDV of changes as a dynamic guide

threshold= df["price_change"].std() * 0.5

def assign_label(change):
    if pd.isna(change):
        return np.nan
    if change > threshold:
        return 2 # UP (BUY)
    elif change < -threshold: 
        return 0 # DOWN (SELL)
    else:
        return 1 # FLAT (HOLD)
    
df["target"]= df["price_change"].apply(assign_label)

df=df.dropna().reset_index(drop=True)

feature_cols=["obi","bid_qty","ask_qty"]
X_raw=df[feature_cols].values
Y_raw= df["target"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Dataset summary after labeling:")
print(df["target"].value_counts(normalize=True))

Dataset summary after labeling:
target
1.0    0.934068
2.0    0.036673
0.0    0.029259
Name: proportion, dtype: float64


In [10]:
def create_hft_sequences(X_data, Y_data, seq_length=50):
    X_seq=[]
    Y_seq=[]

    for i in range(len(X_data)- seq_length):
        # Context window: past 'seq_length' ticks
        X_seq.append(X_data[i:i+seq_length])
        # Target: the label corresponding to the final tick in that context
        Y_seq.append(Y_data[i + seq_length - 1])

    return torch.tensor(np.array(X_seq), dtype=torch.float32), torch.tensor(np.array(Y_seq), dtype= torch.long)

SEQ_LENGTH = 50

X_tensor, Y_tensor = create_hft_sequences(X_scaled, Y_raw, seq_length=SEQ_LENGTH)

#To avoid look ahead leakage
split_idx= int(len(X_tensor) * 0.8)
X_train , X_test=X_tensor[:split_idx], X_tensor[split_idx:]
Y_train, Y_test= Y_tensor[:split_idx], Y_tensor[split_idx:]

print(f"X_train_shape: {X_train.shape}, Y_train_shape: {Y_train.shape}")


X_train_shape: torch.Size([3952, 50, 3]), Y_train_shape: torch.Size([3952])


In [12]:
class MambaHFTClassifier(nn.Module):
    def __init__(self,mamba_core_block, d_model, num_classes=3):
        super().__init__()

        self.mamba=mamba_core_block

        # Linear Layer to project features up if needed
        self.input_projection = nn.Linear(3, d_model)

        # Classifcation Head 
        # the final step's hidden state ad predicts [Down, Flat ,  Up]
        self.classification_head=nn.Linear(d_model, num_classes)

    def foward(self,x):
        # shape: (Batch, Sequence_length, Input_features) -> (B,50,3)
        x= self.input_projection(x) # converting to (B, 50, d_model)
        # passing sequence through the selective state space updates
        mamba_out= self.mamba(x) # (B, 50, d_model)

        final_step_state= mamba_out[:,-1,:] # shape: (B, d_model)

        # Output unnormalized logits for CrossEntropyLoss
        logits = self.classification_head(final_step_state) # Shape: (B,3)
        return logits 

In [22]:
BATCH_SIZE= 64 
EPOCHS = 30 
LEARNING_RATE = 0.001
D_MODEL = 16
D_STATE= 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_dataset = TensorDataset(X_train, Y_train)
train_loader= DataLoader(train_dataset,batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(X_test,Y_test)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

# I carried it from the earlier project done 
class SelectiveSSM(nn.Module):
    def __init__(self,d_model: int, d_state: int, d_rank: int):
        """
        Selective state space Model (Mamba core engine)
        Args: 
            d_model-features
            d_state- size of the internal latent state per feature
            d_rank: Low rank expansion dimension for computing Delta """
        
        super().__init__()
        self.d_model=d_model
        self.d_state=d_state

        # Initializing the continous system matrix A (HiPPO-inspired real diagonal)
        # I need negative values to ensure a stable, decaying system over time
        # shape: (D,N)
        A_init= torch.arange(1, d_state+ 1 , dtype=torch.float32).repeat(d_model,1)
        self.A_log=nn.Parameter(torch.log(A_init))

        #Linear projection layers to make parameters data-dependent 
        self.x_proj=nn.Linear(d_model,d_rank+ d_state+ d_state, bias=False)

        #Project from low_rank back up to d_model for the step_size Delta 
        self.dt_proj=nn.Linear(d_rank , d_model, bias=True)

        # initialzing dt_poj bias to step-size for reasonable initialization
        nn.init.constant_(self.dt_proj.bias, 0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Foward pass processing a macroeconomic panel sequence.

        Args:
            x: Input tensor of shape (B,L,D)
                where B = Countries , L= Quarters, D= Macro features
        
        Returns:
            y: Output tensor of shape (B,L,D)
        """
        B,L,D = x.shape
        N= self.d_state

        # Retrive continous martix A -> shape: (D,N)
        A= -torch.exp(self.A_log)
        
        projected= self.x_proj(x) # shape should be (B,L,d_rank+N+N)

        dt_rank_projected, B_t,C_t= torch.split(projected,
                                                [projected.shape[-1]-2 * N,N,N],
                                                dim=-1)
        # B_t shape: (B, L,N), C_t shape: (B,L,N)

        # computing the dynamic time-step delta

        dt = F.softplus(self.dt_proj(dt_rank_projected))

        # run the selective scan (secquential over the timeline L)
        # Initialize hidden latent state buffer h to zero vector -> shape: (B,D,N)

        h = torch.zeros(B,D,N , device=x.device)
        outputs=[]

        for t in range(L):

            x_t = x[:,t,:]    
            dt_t= dt[:,t,:]
            B_curr = B_t[:,t,:]
            C_curr = C_t[:,t,:]

            # Discretization via Zero Order Hold (ZOH)  ---
            # compute element wise continous expansions using broadcasting 
            # dt_t is (B,D,1) * A is (1,D,N) -> dA is (B,D,N)
            dA= dt_t.unsqueeze(-1)* A.unsqueeze(0)
            A_bar= torch.exp(dA) #shape: (B,D,N)

            # computing B_bar
            # dt_t is (B,D,1)* B_curr is (B,1,N) -> dB is (B,D,N)
            dB= dt_t.unsqueeze(-1) * B_curr.unsqueeze(1)

            # Using the simplified stable approximation for small values
            B_bar= (torch.exp(dA)-1)/ A.unsqueeze(0) * B_curr.unsqueeze(1)
            # Handle possible divisio by zero where elements of A are extremely
            B_bar= torch.where(torch.isnan(B_bar), dB, B_bar)

            #  ----state space evolution ----
            # Update the latent running memory state matrix
            h= A_bar * h + B_bar * x_t.unsqueeze(-1)

            #   --- output Projection ---
            # construct the current quarter output map via C_curr dot-product contraction
            # h is (B,D,N), C_curr is (B,N) -> Output y_curr shape: (B,D)
            y_curr= torch.sum(h * C_curr.unsqueeze(1), dim=-1)
            outputs.append(y_curr)
        
        # Stack sequentially gathered steps back into standard timeline representation 
        y = torch.stack(outputs, dim=1) # Shape: (B,L,D)
        return y
    
    

# reuisng the custom Mamba Engine architecture cofigured as a wrapper
class MambaHFTClassifier(nn.Module):
    def __init__(self,d_model: int , d_state: int, d_rank: int, num_classes=3):
        super().__init__()

        self.input_projection = nn.Linear(3, d_model)

        self.mamba= SelectiveSSM(
            d_model=d_model,
            d_state=d_state,
            d_rank=d_rank
        )

        self.classification_head=nn.Sequential(
            nn.Dropout(0.2), #prevent overfitting the noise
            nn.Linear(d_model,num_classes)
        )
    
    def forward(self,x):
        x=self.input_projection(x)

        mamba_out=self.mamba(x)

        final_step_state=mamba_out[:,-1,:]

        logits = self.classification_head(final_step_state)

        return logits

Using device: cpu


In [25]:
model=MambaHFTClassifier(d_model=D_MODEL,d_state=D_STATE,d_rank=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer=optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=0.01)

print("\n Starting High-Frequency Mamba Training")
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss=0.0
    correct_preds = 0
    total_preds = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        # foward Pass
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        #  Backward pass

        optimizer.zero_grad()
        loss.backward()
        # optimizer.backward()
        optimizer.step()

        # Metrics tracking 
        running_loss += loss.item() * batch_x.size(0)
        _, predictions = torch.max(logits,1)
        correct_preds += (predictions==batch_y).sum().item()
        total_preds+=batch_y.size(0)

    epoch_loss= running_loss/total_preds
    epoch_acc= (correct_preds/total_preds)*100

    if epoch % 5== 0 or epoch==1:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_acc:.2f}%")


model.eval()
test_correct = 0 
test_total= 0
all_preds=[]
all_targets =[]

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y= batch_x.to(device), batch_y.to(device)
        logits= model(batch_x)

        _, predictions = torch.max(logits, 1)
        test_correct += (predictions == batch_y).sum().item()
        test_total += batch_y.size(0)

        all_preds.extend(predictions.cpu().numpy())
        all_targets.extend(batch_y.cpu().numpy())

final_test_acc = (test_correct/test_total) * 100
print(f"Test Accuracy: {final_test_acc:.2f}%")

unique, counts=np.unique(all_targets, return_counts=True)
majority_class_baseline=(max(counts)/len(all_targets)) * 100
print(f"Naively guessing the most frequency class gets: {majority_class_baseline:.2f}")


 Starting High-Frequency Mamba Training


NameError: name 'F' is not defined